# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset via the [mlcroissant](https://mlcommons.github.io/croissant/python/quickstart/) library.

### Dataset Source
The dataset source is provided via a [Croissant schema JSON-LD URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json). This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples extracted from extracellular vesicles isolated from stimulated human mast cells.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd
# Define the dataset Croissant schema URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadataprint(f"Dataset Title: {metadata.name}")print(f"Description: {metadata.description}")print(f"Version: {metadata.version}")print(f"License: {metadata.license}")print("\nUse Cases:")for uc in getattr(metadata, 'dataUseCases', []):    print(f"- {uc}")

## 2. Data Overview
Review available record sets, their fields, and columns as defined by their `@id`s.

A record set is a logical table (such as a file or a structured list) within the dataset. Each record set defines fields and columns, each with their own `@id` for reference.

In [ ]:
# List all record sets in the dataset and their schemas
record_sets = dataset.record_sets
print(f"Total record sets in schema: {len(record_sets)}\n")rs_schema = {}for rs in record_sets:    print(f"RecordSet @id: {rs['@id']}")    name = rs.get('name', '(no name)')    print(f"  Name: {name}")    fields = rs.get('field', []) or []    if not isinstance(fields, list):        fields = [fields]    print(f"  Fields/columns (@id):")    for field in fields:        print(f"    - {field.get('@id', field)}")    rs_schema[rs['@id']] = fields    print()

## 3. Data Extraction
Load data from record sets into DataFrames for analysis. Use the record set and field `@id`s as identified above.

In [ ]:
# For this dataset, the main record set is likely to be '@id': 'cr:RecordSet/protein-table', corresponding to the protein table.
# If multiple record sets exist, you can process several; here select the main one for demonstration.
main_record_set_id = Nonefor rs in dataset.record_sets:
    # Select the first record set (change logic to select the main table explicitly if needed)
    main_record_set_id = rs['@id']
    break

print(f"Extracting data from RecordSet: {main_record_set_id}")
# Load records into a DataFrame
records_iterator = dataset.records(record_set=main_record_set_id)records = list(records_iterator)df = pd.DataFrame(records)print(f"Loaded {len(df)} records. Columns:")print(df.columns.tolist())df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. Here we: 
- Select a numeric field (e.g., peptide count or molecular weight)
- Filter records (e.g., MW > 20000)
- Normalize values
- (Optionally) Group by a key attribute (e.g., sample type or accession)

Refer to columns by their `@id` as described in the schema overview above.

In [ ]:
# Identify a numeric field for demonstration. Replace with the actual @id (column name) from your schema. For example, 'cr:Field/molecular_weight'.
numeric_field = None
for col in df.columns:
    if 'weight' in col.lower() or 'mw' in col.lower():
        numeric_field = col
        break
if not numeric_field:
    numeric_field = df.columns[0]  # fallback

print(f"Using numeric field: {numeric_field}")

# Make sure the field is numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter: keep only entries with MW > 20000 (example threshold)
threshold = 20000
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
print(filtered_df[[numeric_field]].head())

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"Normalized {numeric_field} (first 5):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optionally, group by accession or another categorical field (@id)
group_field = None
for col in df.columns:
    if 'accession' in col.lower():
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {group_field} (first 5):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the main numeric field in the filtered data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field} (Filtered > {threshold})')
plt.show()

## 6. Conclusion
- Loaded FAIR² mass spectrometry dataset metadata and protein table records using `mlcroissant`.
- Explored record set schemas and extracted main table into a DataFrame.
- Filtered and normalized a numeric field (e.g., molecular weight), and grouped results where useful.
- Visualized the distribution of protein molecular weights in extracellular vesicle samples.

This notebook provides a foundation for protein biomarker, modification, and abundance analyses. See the dataset's Croissant schema and `mlcroissant` [documentation](https://mlcommons.github.io/croissant/python/quickstart/) for further data exploration options.